In [ ]:
import os
import pandas as pd
from tabulate import tabulate

In [5]:
tags = pd.read_csv("..\\Data\\03-01_회전기계_신호_회전기계태그목록.csv")
df = pd.read_csv("..\\Data\\03-01_회전기계_신호_진동추세.csv",encoding = "utf-8")

pd.set_option("display.width", 1000)

In [6]:
print(tags.loc[tags["equipment"]=="1번 모터"])

             tag equipment physical_qty indicator summary  unit direction  install_location
0    MTR01_VIB_H     1번 모터           진동        속도     RMS  mm/s        수평       구동측 베어링 하우징
1    MTR01_VIB_V     1번 모터           진동        속도     RMS  mm/s        수직       구동측 베어링 하우징
2    MTR01_VIB_A     1번 모터           진동        속도     RMS  mm/s       축방향       구동측 베어링 하우징
3  MTR01_VIB_ACC     1번 모터           진동       가속도    PEAK     g        수평       구동측 베어링 하우징
4  MTR01_CURRENT     1번 모터           전류        전류     순시값     A      해당없음     모터 제어반 전류 변성기
5     MTR01_TEMP     1번 모터           온도        온도     순시값  degC      해당없음  구동측 베어링 매입 측온저항체
6      MTR01_RPM     1번 모터          회전수       회전수     순시값   rpm      해당없음       축 단부 회전수 센서


In [24]:
# 진동의 정상 범위 정하기
# 맨 앞 기준으로 20일 구간을 정상 기간으로 볼 것

normal = df.head()
MTR = list(df.columns[df.columns.str.startswith("MTR")])
PMP = list(df.columns[df.columns.str.startswith("PMP")])
print(MTR)
print(PMP)

print(normal[MTR].agg(["min", "max"]))
print(normal[PMP].agg(["min", "max"]))

['MTR01_VIB_H', 'MTR01_VIB_V', 'MTR01_VIB_A', 'MTR01_VIB_ACC', 'MTR01_CURRENT', 'MTR01_TEMP', 'MTR01_RPM']
['PMP01_VIB_H', 'PMP01_VIB_V', 'PMP01_VIB_A', 'PMP01_VIB_ACC']
     MTR01_VIB_H  MTR01_VIB_V  MTR01_VIB_A  MTR01_VIB_ACC  MTR01_CURRENT  MTR01_TEMP  MTR01_RPM
min          1.8          1.4          0.7           0.52           62.0        48.0       1780
max          2.0          1.5          0.8           0.55           64.0        49.0       1780
     PMP01_VIB_H  PMP01_VIB_V  PMP01_VIB_A  PMP01_VIB_ACC
min          2.0          1.6          0.9           0.60
max          2.2          1.7          1.0           0.63


In [23]:
# 실습 - 펌프 극단값 출력해보기

pump = tags[tags["equipment"] == "1번 펌프"]

pump.info()
pump.head()



<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 7 to 13
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   tag               7 non-null      str  
 1   equipment         7 non-null      str  
 2   physical_qty      7 non-null      str  
 3   indicator         7 non-null      str  
 4   summary           7 non-null      str  
 5   unit              7 non-null      str  
 6   direction         7 non-null      str  
 7   install_location  7 non-null      str  
dtypes: str(8)
memory usage: 580.0 bytes


,tag,equipment,physical_qty,indicator,summary,unit,direction,install_location
7,PMP01_VIB_H,1번 펌프,진동,속도,RMS,mm/s,수평,구동측 베어링 하우징
8,PMP01_VIB_V,1번 펌프,진동,속도,RMS,mm/s,수직,구동측 베어링 하우징
9,PMP01_VIB_A,1번 펌프,진동,속도,RMS,mm/s,축방향,구동측 베어링 하우징
10,PMP01_VIB_ACC,1번 펌프,진동,가속도,PEAK,g,수평,구동측 베어링 하우징
11,PMP01_CURRENT,1번 펌프,전류,전류,순시값,A,해당없음,펌프 제어반 전류 변성기


In [41]:
Q1 = df[PMP].quantile(0.25)
print("4분위수 Q1값:\n", Q1)
print()
Q3 = df[PMP].quantile(0.75)
print("4분위수 Q3값:\n", Q3)

IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df[PMP] < lower_bound) | (df[PMP] > upper_bound)]

print(outliers)

4분위수 Q1값:
 PMP01_VIB_H      2.1
PMP01_VIB_V      1.6
PMP01_VIB_A      0.9
PMP01_VIB_ACC    0.6
Name: 0.25, dtype: float64

4분위수 Q3값:
 PMP01_VIB_H      2.325
PMP01_VIB_V      1.800
PMP01_VIB_A      1.600
PMP01_VIB_ACC    0.620
Name: 0.75, dtype: float64
   date  MTR01_VIB_H  MTR01_VIB_V  MTR01_VIB_A  MTR01_VIB_ACC  MTR01_CURRENT  MTR01_TEMP  MTR01_RPM  PMP01_VIB_H  PMP01_VIB_V  PMP01_VIB_A  PMP01_VIB_ACC
0   NaN          NaN          NaN          NaN            NaN            NaN         NaN        NaN          NaN          NaN          NaN            NaN
1   NaN          NaN          NaN          NaN            NaN            NaN         NaN        NaN          NaN          NaN          NaN            NaN
2   NaN          NaN          NaN          NaN            NaN            NaN         NaN        NaN          NaN          NaN          NaN            NaN
3   NaN          NaN          NaN          NaN            NaN            NaN         NaN        NaN          NaN          NaN      

csv 60행 PMP01_VIB_H가 Outliers 되어 있음.

In [55]:
# 1번 모터의 회전수 컬럼 이름 / 1780, 1450 종류의 숫자가 각각 몇 번 찍히는지 확인해보기
print(df["MTR01_RPM"].value_counts())

MTR01_RPM
1780    56
1450     4
Name: count, dtype: int64


In [53]:
print(
    df.loc[df["MTR01_RPM"] == 1780, ["date", "MTR01_VIB_H", "MTR01_VIB_ACC", "MTR01_RPM"]].head(4)
)

         date  MTR01_VIB_H  MTR01_VIB_ACC  MTR01_RPM
0  2026-01-01          1.8           0.52       1780
1  2026-01-02          1.9           0.54       1780
2  2026-01-03          1.8           0.53       1780
3  2026-01-04          2.0           0.55       1780


### **강사님 코드**

In [58]:
import pandas as pd

tags = pd.read_csv("..\\Data\\03-01_회전기계_신호_회전기계태그목록.csv")
df = pd.read_csv("..\\Data\\03-01_회전기계_신호_진동추세.csv")

# 1번 모터에 대한 태그 목록
# ["tag", "indicator", "summary", "unit", "direction"]
print(tags.loc[tags["equipment"]=="1번 모터", 
               ["tag", "indicator", "summary", "unit", "direction"]])
'''
             tag indicator summary  unit direction
0    MTR01_VIB_H        속도     RMS  mm/s        수평
1    MTR01_VIB_V        속도     RMS  mm/s        수직
2    MTR01_VIB_A        속도     RMS  mm/s       축방향
3  MTR01_VIB_ACC       가속도    PEAK     g        수평
4  MTR01_CURRENT        전류     순시값     A      해당없음
5     MTR01_TEMP        온도     순시값  degC      해당없음
'''

# 진동의 정상 범위 정하기
# 맨 앞 기준으로 20일 구간을 정상 기간으로 볼 것
MTR = ["MTR01_VIB_H", "MTR01_VIB_V", "MTR01_VIB_A", "MTR01_VIB_ACC"]
PMP = ["PMP01_VIB_H", "PMP01_VIB_V", "PMP01_VIB_A", "PMP01_VIB_ACC"]
normal = df.head(20)

print(normal[MTR].agg(["min", "max"])) # 극단값(최대, 최솟값) 출력
'''
     MTR01_VIB_H  MTR01_VIB_V  MTR01_VIB_A  MTR01_VIB_ACC
min          1.7          1.3          0.7           0.51
max          2.1          1.6          0.8           0.56
'''
print(normal[PMP].agg(["min", "max"])) # 극단값(최대, 최솟값) 출력
'''
     PMP01_VIB_H  PMP01_VIB_V  PMP01_VIB_A  PMP01_VIB_ACC
min          2.0          1.6          0.9           0.60
max          2.2          1.7          1.0           0.63
'''

def first_over(col):
    """정상 구간 최댓값을 처음 넘어선 행의 순서를 반환합니다."""
    limit = normal[col].max() # 20일구간(normal) 의 최댓값을 정상 범위로 설정
    over = df.index[df[col]>limit] # 정상 범위를 넘는 index를 불러온다.
    # print(over)
    # Index([38, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57,
    #   58, 59],
    #  dtype='int64')
    # print(over[0]) #38

    return int(over[0]) + 1 if len(over) else None

print("--------------")
print("[1번 모터]")
for c in MTR:
    print(c, first_over(c))
'''
[1번 모터]
MTR01_VIB_H 39
MTR01_VIB_V 44
MTR01_VIB_A None
MTR01_VIB_ACC 22
'''

print("--------------")
print("[1번 펌프]")
for c in PMP:
    print(c, first_over(c))
'''
PMP01_VIB_H 39
PMP01_VIB_V 38
PMP01_VIB_A 35
PMP01_VIB_ACC None
'''

## 모터의 전류와 온도

print("--------------")
print("[1번 모터: 전류와 온도]")
for c in ["MTR01_CURRENT","MTR01_TEMP"]:
    print(c, first_over(c))

# 전류와 온도가 이상반응에 대해서 늦게 반응한다.
'''
[1번 모터: 전류와 온도]
MTR01_CURRENT 49
MTR01_TEMP 53
'''

##### 모터1번의 회전수
# 모터1번 회전수의 컬럼 이름
# 1780, 1450 종류의 숫자가 각각 몇 번 찍히는지
print(df["MTR01_RPM"].value_counts())
'''
MTR01_RPM
1780    56
1450     4
'''
print("==========")
print(df.loc[
    df["MTR01_RPM"]==1780, ["date", "MTR01_VIB_H", "MTR01_VIB_ACC", "MTR01_RPM"]
].head(4))
print(df.loc[
    df["MTR01_RPM"]==1450, ["date", "MTR01_VIB_H", "MTR01_VIB_ACC", "MTR01_RPM"]
])

'''
         date  MTR01_VIB_H  MTR01_VIB_ACC  MTR01_RPM
0  2026-01-01          1.8           0.52       1780
1  2026-01-02          1.9           0.54       1780
2  2026-01-03          1.8           0.53       1780
3  2026-01-04          2.0           0.55       1780
          date  MTR01_VIB_H  MTR01_VIB_ACC  MTR01_RPM
29  2026-01-30          1.3           0.61       1450
30  2026-01-31          1.3           0.63       1450
31  2026-02-01          1.4           0.67       1450
32  2026-02-02          1.3           0.68       1450
'''
# 설비의 변화가 아닌 회전수(RPM)의 변화로 진동수가 변경되었다.

             tag indicator summary  unit direction
0    MTR01_VIB_H        속도     RMS  mm/s        수평
1    MTR01_VIB_V        속도     RMS  mm/s        수직
2    MTR01_VIB_A        속도     RMS  mm/s       축방향
3  MTR01_VIB_ACC       가속도    PEAK     g        수평
4  MTR01_CURRENT        전류     순시값     A      해당없음
5     MTR01_TEMP        온도     순시값  degC      해당없음
6      MTR01_RPM       회전수     순시값   rpm      해당없음
     MTR01_VIB_H  MTR01_VIB_V  MTR01_VIB_A  MTR01_VIB_ACC
min          1.7          1.3          0.7           0.51
max          2.1          1.6          0.8           0.56
     PMP01_VIB_H  PMP01_VIB_V  PMP01_VIB_A  PMP01_VIB_ACC
min          2.0          1.6          0.9           0.60
max          2.2          1.7          1.0           0.63
--------------
[1번 모터]
MTR01_VIB_H 39
MTR01_VIB_V 44
MTR01_VIB_A None
MTR01_VIB_ACC 22
--------------
[1번 펌프]
PMP01_VIB_H 39
PMP01_VIB_V 38
PMP01_VIB_A 35
PMP01_VIB_ACC None
--------------
[1번 모터: 전류와 온도]
MTR01_CURRENT 49
MTR01_TEMP 53
MTR01_RP

'\n         date  MTR01_VIB_H  MTR01_VIB_ACC  MTR01_RPM\n0  2026-01-01          1.8           0.52       1780\n1  2026-01-02          1.9           0.54       1780\n2  2026-01-03          1.8           0.53       1780\n3  2026-01-04          2.0           0.55       1780\n          date  MTR01_VIB_H  MTR01_VIB_ACC  MTR01_RPM\n29  2026-01-30          1.3           0.61       1450\n30  2026-01-31          1.3           0.63       1450\n31  2026-02-01          1.4           0.67       1450\n32  2026-02-02          1.3           0.68       1450\n'